## Spark Core Questions

### 1. Loading and Transforming Data:
Question: Load the "airports.csv" dataset into a Data Frame. Select the columns "Name", "City",
and "Country” and filter the Data Frame to include only airports in Canada. Show the first 10
rows of the filtered Data Frame.


In [0]:
# Load the data into dataframe
airport_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("dbfs:/FileStore/shared_uploads/200573774@student.georgianc.on.ca/aiport.csv")

In [0]:
airport_df_schema = airport_df.schema
print(airport_df.printSchema())

root
 |-- Column1: integer (nullable = true)
 |-- Column2: string (nullable = true)
 |-- Column3: string (nullable = true)
 |-- Column4: string (nullable = true)
 |-- Column5: string (nullable = true)
 |-- Column6: string (nullable = true)
 |-- Column7: double (nullable = true)
 |-- Column8: double (nullable = true)
 |-- Column9: integer (nullable = true)
 |-- Column10: double (nullable = true)
 |-- Column11: string (nullable = true)
 |-- Column12: string (nullable = true)
 |-- Column13: string (nullable = true)
 |-- Column14: string (nullable = true)

None


In [0]:
airport_df.show(5)

+-------+--------------------+------------+----------------+-------+-------+------------+-----------+-------+--------+--------+--------------------+--------+-----------+
|Column1|             Column2|     Column3|         Column4|Column5|Column6|     Column7|    Column8|Column9|Column10|Column11|            Column12|Column13|   Column14|
+-------+--------------------+------------+----------------+-------+-------+------------+-----------+-------+--------+--------+--------------------+--------+-----------+
|      1|      Goroka Airport|      Goroka|Papua New Guinea|    GKA|   AYGA|-6.081689835|145.3919983|   5282|    10.0|       U|Pacific/Port_Moresby| airport|OurAirports|
|      2|      Madang Airport|      Madang|Papua New Guinea|    MAG|   AYMD|-5.207079887|145.7890015|     20|    10.0|       U|Pacific/Port_Moresby| airport|OurAirports|
|      3|Mount Hagen Kagam...| Mount Hagen|Papua New Guinea|    HGU|   AYMH|-5.826789856|144.2960052|   5388|    10.0|       U|Pacific/Port_Moresby| a

In [0]:
# Transforming and seleting only the Name, City, Country columns  
rename_dict = { "Column2": "Name", "Column3": "City", "Column4": "Country","Column9": "Altitude", "Column12": "Timezone"}

for old_col, new_col in rename_dict.items():
    airport_df = airport_df.withColumnRenamed(old_col, new_col)

airport_df = airport_df.select(["Name","City","Country", "Altitude", "Timezone"])


In [0]:
# Filtering the Data Frame to include only airports in Canada
airport_df_filtered = airport_df.select(["Name","City","Country"]).filter(airport_df.Country == "Canada")
airport_df_filtered.show(10)

+--------------------+------------------+-------+
|                Name|              City|Country|
+--------------------+------------------+-------+
|Sault Ste Marie A...|Sault Sainte Marie| Canada|
|Winnipeg / St. An...|          Winnipeg| Canada|
|Halifax / CFB She...|           Halifax| Canada|
| St. Anthony Airport|       St. Anthony| Canada|
|Tofino / Long Bea...|            Tofino| Canada|
|    Kugaaruk Airport|         Pelly Bay| Canada|
| Baie Comeau Airport|       Baie Comeau| Canada|
|      CFB Bagotville|        Bagotville| Canada|
|  Baker Lake Airport|        Baker Lake| Canada|
|Campbell River Ai...|    Campbell River| Canada|
+--------------------+------------------+-------+
only showing top 10 rows



### 2. Counting Airports by Country:
Question: Count the number of airports in each country and show the top 5 countries with the
most airports.


In [0]:
# Counting the number of airports in each country and showing the top 5
most_airports = airport_df.groupBy("Country").count().sort("Count", ascending=False)
most_airports.show(5)

+-------------+-----+
|      Country|count|
+-------------+-----+
|United States| 1512|
|       Canada|  430|
|    Australia|  334|
|       Russia|  264|
|       Brazil|  264|
+-------------+-----+
only showing top 5 rows



### 3. Airport Altitude Statistics:
Question: Calculate the minimum, maximum, and average altitude of airports. Show the results.


In [0]:
airport_df.columns

Out[44]: ['Name', 'City', 'Country', 'Altitude', 'Timezone']

In [0]:
from pyspark.sql import functions as func
from pyspark.sql.types import IntegerType
airport_df = airport_df.withColumn("Altitude", func.col("Altitude").cast(IntegerType()))

In [0]:
# Calculate the minimum, maximum, and average altitude of airports
airport_df.select(
    func.mean("Altitude").alias("mean_altitude"),
    func.min("Altitude").alias("min_altitude"),
    func.max("Altitude").alias("max_altitude"),
).show()


+-----------------+------------+------------+
|    mean_altitude|min_altitude|max_altitude|
+-----------------+------------+------------+
|1015.873343725643|       -1266|       14472|
+-----------------+------------+------------+



## Spark SQL Questions

### 1. Creating a SQL View:
Question: Create a temporary SQL view called "airports" from the "airports.csv" Data Frame.
Write a SQL query to select the names and cities of airports located in France. Show the first 10
results.


In [0]:
# Creating SQL view airports
airport_df.createOrReplaceTempView("airports")

In [0]:
# selecting the names and cities of airports located in France
result = spark.sql("SELECT Name, City, Country from airports where Country = 'France'")
result.show()

+--------------------+-----------------+-------+
|                Name|             City|Country|
+--------------------+-----------------+-------+
|Calais-Dunkerque ...|           Calais| France|
|P�ronne-Saint-Que...|          Peronne| France|
|Nangis-Les Loges ...|           Nangis| France|
|Bagnoles-de-l'Orn...|Bagnole-de-l'orne| France|
| Albert-Bray Airport|           Albert| France|
|Le Touquet-C�te d...|      Le Tourquet| France|
|Valenciennes-Dena...|     Valenciennes| France|
|Amiens-Glisy Airport|           Amiens| France|
|Agen-La Garenne A...|             Agen| France|
|Cazaux (BA 120) A...|           Cazaux| France|
|Bordeaux-M�rignac...|         Bordeaux| France|
|Bergerac-Roumani�...|         Bergerac| France|
|Toulouse-Francaza...|         Toulouse| France|
|Cognac-Ch�teauber...|           Cognac| France|
|Poitiers-Biard Ai...|         Poitiers| France|
|Montlu�on-Gu�ret ...| Montlucon-gueret| France|
|     Limoges Airport|          Limoges| France|
|Mont-de-Marsan (B..

### 2. Counting Airports by Time zone:
Question: Write a SQL query to count the number of airports in each time zone. Show the results
ordered by the count in descending order.

In [0]:
result = spark.sql("Select Timezone, count(*) as Total_airports from airports group by Timezone order by Total_airports desc")
result.show()

+-------------------+--------------+
|           Timezone|Total_airports|
+-------------------+--------------+
|                 \N|          1021|
|   America/New_York|           449|
|    America/Chicago|           345|
|      Europe/Berlin|           222|
|       Europe/Paris|           208|
|      Asia/Shanghai|           187|
|  America/Anchorage|           177|
|America/Los_Angeles|           166|
|      Europe/London|           155|
|    America/Toronto|           133|
|      Asia/Calcutta|           131|
|     America/Denver|           118|
|         Asia/Tokyo|           113|
|Africa/Johannesburg|            94|
|  America/Sao_Paulo|            93|
| Australia/Brisbane|            81|
|        Asia/Tehran|            78|
|   Europe/Stockholm|            76|
|      Europe/Moscow|            75|
|        Europe/Rome|            74|
+-------------------+--------------+
only showing top 20 rows



### 3. Airport Altitude Analysis:
Question: Write a SQL query to find the airports with an altitude greater than 2000 meters. Select
the airport name, city, and altitude. Show the first 10 results.


In [0]:
result = spark.sql("Select Name, City, Altitude from airports where Altitude > 2000 limit 10")
result.show()

+--------------------+------------+--------+
|                Name|        City|Altitude|
+--------------------+------------+--------+
|      Goroka Airport|      Goroka|    5282|
|Mount Hagen Kagam...| Mount Hagen|    5388|
|  Coronation Airport|  Coronation|    2595|
|     Burwash Airport|     Burwash|    2647|
|   Princeton Airport|   Princeton|    2298|
|  Dease Lake Airport|  Dease Lake|    2600|
|Dawson Creek Airport|Dawson Creek|    2148|
|Edmonton Internat...|    Edmonton|    2373|
|       Edson Airport|       Edson|    3043|
|  Kindersley Airport|  Kindersley|    2277|
+--------------------+------------+--------+



## Spark Streaming Question

### Streaming from a Directory:
Question: Set up a Spark streaming job to read new CSV files from a directory (e.g.,
/FileStore/tables/streaming). Process the stream to count the number of airports in each country
and write the results to the console.


In [0]:
# Our file is uploaded to the databricks shard file store
input_path = "dbfs:/FileStore/shared_uploads/200573774@student.georgianc.on.ca/"
df = spark.readStream.format("csv").option("header", "true").schema(airport_df_schema).load(input_path)

#Column 4 is the country
country_airports_count = df.groupBy("Column4").count()

query = country_airports_count.writeStream.outputMode("complete").format("console").start()

query.awaitTermination()
